# Phase 7: YOLOv12 Optimization Study (Part 1 - Training)

This notebook runs multiple YOLO training experiments to optimize Stage 1 recall. It focuses on Image Resolution, Training Strategies, and Augmentation. 

**Note: This notebook uses ONLY the training and validation sets.**

In [9]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Install dependencies
!pip install ultralytics

In [11]:
import os
import sys

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/sem_defect_project


In [12]:
# We will use the data directly from Google Drive as requested.
data_yaml_path = 'dataset_yolo_single_class/data.yaml'

print(f"Using dataset configuration from: {data_yaml_path}")

Using dataset configuration from: dataset_yolo_single_class/data.yaml


## 1. EXP-05: Resolution 640x640
Tests if a moderate increase in resolution improves small object detection.

In [13]:
from ultralytics import YOLO

# Initialize YOLOv12 (or YOLO11 fallback)
try:
    model_640 = YOLO('yolov12n.pt')
except Exception:
    print("YOLOv12 weights not found, using YOLO11n fallback.")
    model_640 = YOLO('yolo11n.pt')

results_640 = model_640.train(
    data=data_yaml_path,
    epochs=40,
    imgsz=640,
    batch=32,
    project='runs/detect/optimization',
    name='EXP-05-Res640'
)


YOLOv12 weights not found, using YOLO11n fallback.
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=

## 2. EXP-06: Resolution 768x768
Tests if pushing resolution even higher yields better recall for microporosities.

In [14]:
try:
    model_768 = YOLO('yolov12n.pt')
except Exception:
    model_768 = YOLO('yolo11n.pt')

results_768 = model_768.train(
    data=data_yaml_path,
    epochs=40,
    imgsz=768,
    batch=16, # Reduced batch size to fit in GPU memory
    project='runs/detect/optimization',
    name='EXP-06-Res768'
)


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=EXP-06-Res768, nbs=64, nms=None, opset=No

## 3. EXP-07: Enhanced Training (100 Epochs, AdamW)
We use the best resolution (e.g., 640) and push training to 100 epochs with AdamW and early stopping to ensure convergence.

In [15]:
# Note: You can change imgsz=640 to 768 if EXP-06 was vastly superior
try:
    model_opt = YOLO('yolov12n.pt')
except Exception:
    model_opt = YOLO('yolo11n.pt')

results_opt = model_opt.train(
    data=data_yaml_path,
    epochs=100,
    patience=20, # Early stopping
    imgsz=640, # Using 640 as a safe baseline, adjust if 768 was better
    batch=32,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.01,
    project='runs/detect/optimization',
    name='EXP-07-AdamW100'
)


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=EXP-07-AdamW100, nbs=64, nms=None, opse

## 4. EXP-08: Conservative Augmentation
We disable aggressive augmentations (Mosaic, MixUp) to preserve SEM textures. We base this on the enhanced training setup.

In [16]:
try:
    model_aug = YOLO('yolov12n.pt')
except Exception:
    model_aug = YOLO('yolo11n.pt')

results_aug = model_aug.train(
    data=data_yaml_path,
    epochs=100,
    patience=20,
    imgsz=640,
    batch=32,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.01,
    mosaic=0.0, # Disable Mosaic
    mixup=0.0,  # Disable Mixup
    degrees=0.0, # Handled offline
    flipud=0.0, # Handled offline
    fliplr=0.0, # Handled offline
    project='runs/detect/optimization',
    name='EXP-08-NoMosaic'
)


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=EXP-08-NoMosaic, nbs=64, nms=None, opse